# Week 11: Advanced Specialized Machine Learning
## Notebook 08: Two-Tower Neural Recommenders & In-Batch Contrastive Loss

### 1. Conceptual Intuition & Theoretical Background
Industrial recommendation systems (e.g. YouTube, TikTok, Netflix) operate in two distinct stages:
1. **Candidate Generation (Retrieval):** Sifting through 100M+ catalog items to find top 1,000 candidates in $<10\text{ms}$.
2. **Ranking:** Heavy Transformer / DLRM scoring on top candidates.

The **Two-Tower Neural Network** makes microsecond retrieval possible:
- **User Tower:** Maps dynamic user context (search history, age, location) to embedding vector:
$$u(x) \in \mathbb{R}^d$$
- **Item Tower:** Maps static item catalog attributes (title, category, tags) to embedding vector:
$$v(y) \in \mathbb{R}^d$$

During offline indexing, all item embeddings $v(y)$ are pre-computed and stored in an Approximate Nearest Neighbor (ANN) index (e.g. FAISS / HNSW).
At inference time, query scoring reduces to a simple dot product:
$$\text{Score}(x, y) = \langle u(x), v(y) \rangle$$

Trained via **In-Batch Negative Softmax Loss**:
$$\mathcal{L} = -\frac{1}{B} \sum_{i=1}^{B} \log \frac{\exp(\langle u_i, v_i \rangle / \tau)}{\sum_{j=1}^{B} \exp(\langle u_i, v_j \rangle / \tau)}$$

---

### 2. Scratch Python Implementation: Two-Tower Contrastive Engine

In [ ]:
import numpy as np

class TwoTowerModelScratch:
    def __init__(self, user_dim, item_dim, embed_dim=16, temperature=0.07):
        self.embed_dim = embed_dim
        self.tau = temperature
        
        # User Tower weights (user_dim -> embed_dim)
        np.random.seed(42)
        self.W_u = np.random.normal(0, 0.1, (user_dim, embed_dim))
        # Item Tower weights (item_dim -> embed_dim)
        self.W_v = np.random.normal(0, 0.1, (item_dim, embed_dim))
        
    def encode_users(self, X_users):
        u = np.dot(X_users, self.W_u)
        # L2 normalize
        return u / np.linalg.norm(u, axis=1, keepdims=True)
        
    def encode_items(self, X_items):
        v = np.dot(X_items, self.W_v)
        # L2 normalize
        return v / np.linalg.norm(v, axis=1, keepdims=True)
        
    def compute_inbatch_loss(self, X_users, X_items):
        batch_size = len(X_users)
        u_emb = self.encode_users(X_users)
        v_emb = self.encode_items(X_items)
        
        # Similarity matrix: (B, B)
        sim_matrix = np.dot(u_emb, v_emb.T) / self.tau
        
        # In-batch positive pairs lie on the diagonal (i == j)
        exp_sim = np.exp(sim_matrix - np.max(sim_matrix, axis=1, keepdims=True))
        probs = exp_sim / np.sum(exp_sim, axis=1, keepdims=True)
        
        loss = -np.mean(np.log(np.diag(probs) + 1e-9))
        return loss, probs

# Synthetic Streaming Batch: 64 users paired with 64 engaged items
batch_size = 64
dim_u = 32
dim_v = 48

X_u_batch = np.random.randn(batch_size, dim_u)
X_v_batch = np.random.randn(batch_size, dim_v)

model = TwoTowerModelScratch(dim_u, dim_v, embed_dim=16, temperature=0.1)
loss_val, prob_matrix = model.compute_inbatch_loss(X_u_batch, X_v_batch)

# Offline Index Retrieval Simulation
catalog_size = 10000
catalog_features = np.random.randn(catalog_size, dim_v)
catalog_embeddings = model.encode_items(catalog_features)

# Single online user query
single_user = X_u_batch[0:1]
user_query_emb = model.encode_users(single_user)

# Cosine retrieval: MIPS (Maximum Inner Product Search)
retrieval_scores = np.dot(catalog_embeddings, user_query_emb.T).squeeze()
top_5_items = np.argsort(-retrieval_scores)[:5]

print(f"Batch Contrastive Loss:               {loss_val:.4f}")
print(f"Catalog Items Indexed in FAISS:       {catalog_size}")
print(f"Top 5 Retrieved Catalog Item IDs:     {top_5_items.tolist()}")
print(f"Top 1 Candidate Dot Product Score:    {retrieval_scores[top_5_items[0]]:.4f}")

### 3. Step-by-Step Code Walkthrough
1. **Tower Decoupling**: Item embeddings are generated completely independently of users and cached offline.
2. **In-Batch Negatives**: Avoids expensive explicit negative sampling; for user $i$, items $j \neq i$ serve as negatives.
3. **MIPS Scoring**: Sub-millisecond ANN search using matrix multiplication.

---

### 4. Industry Applications & Trade-offs
- **YouTube Candidate Retrieval:** Scales retrieval over millions of videos in under 15 milliseconds.
- **Trade-off:** Incapable of modeling complex cross-feature interactions (e.g. user age interacting with video category) until the downstream ranking stage.

---

### 5. Top Interview Questions & Answers
**Q1: Why is in-batch negative sampling biased towards popular items?**
> **Answer:** Because popular items are interacted with frequently, they appear as positive targets in many training batches. As a result, they are penalized as negative examples much more often than niche items. This is corrected via log-frequency sampling corrections (e.g. subtracting $\log P(\text{item})$ from logits).

**Q2: What is the computational complexity of candidate retrieval in a Two-Tower system?**
> **Answer:** Query embedding generation takes $O(d_{\text{user}} \cdot d_{\text{embed}})$. Retrieving top $K$ items from an ANN index (such as HNSW) takes $O(\log N)$ where $N$ is catalog size.